# Bronze electricity: 2025 onward; only missing dates and SE1-SE4

HTTP requests are attempted once. Configure retries on the Databricks notebook task (for example, 2 retries with a 60-second interval). ADF Job activity retries should be 0. Failures include the target table, date and price area.


In [ ]:
import json
import time
from datetime import date, datetime, timedelta, timezone
from functools import reduce
from operator import mul
from zoneinfo import ZoneInfo

import requests
from pyspark.sql.types import (
    DateType, IntegerType, StringType, StructField, StructType, TimestampType
)

CATALOG = "db_labb2"
AREAS = ("SE1", "SE2", "SE3", "SE4")
SCB_BASE = "https://statistikdatabasen.scb.se/api/v2/tables"
PRICE_BASE = "https://www.elprisetjustnu.se/api/v1/prices"

def http_session():
    """Create a session without HTTP-level retries; Jobs owns task retries."""
    return requests.Session()


def response_json(session, url, params=None, *, context="API request"):
    """Request once and raise a contextual failure for Databricks/ADF."""
    try:
        response = session.get(url, params=params, timeout=(10, 90))
        response.raise_for_status()
    except requests.RequestException as exc:
        message = (
            f"{context}: API request failed; "
            f"{type(exc).__name__}: {exc}; url={url}; "
            "HTTP-level retries are disabled; retries are managed by Databricks task settings."
        )
        print(f"[API FAILED] {message}", flush=True)
        raise RuntimeError(message) from exc
    try:
        return response.json()
    except ValueError as exc:
        message = (
            f"{context}: HTTP {response.status_code}, but response is not valid JSON; "
            f"{exc}; url={url}"
        )
        print(f"[API FAILED] {message}", flush=True)
        raise RuntimeError(message) from exc


print("Target catalog:", CATALOG)


In [ ]:
PRICE_SCHEMA = StructType([
    StructField("source", StringType(), False),
    StructField("price_date", DateType(), False),
    StructField("price_area", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("fetched_at_utc", TimestampType(), False),
    StructField("payload_json", StringType(), False),
])

def days_inclusive(start, end):
    """Yield every calendar date from start through end, inclusive."""
    current = start
    while current <= end:
        yield current
        current += timedelta(days=1)

def validate_price_payload(payload, day, area):
    """Check interval length and continuous coverage of one Swedish local day.

    Prices are hourly before 2025-10-01 and quarter-hourly thereafter.
    UTC comparisons also handle the 23-hour and 25-hour daylight-saving days.
    The source has one incorrect time_end at the 2025 autumn clock change;
    retain the raw payload and validate that one interval by its start time.
    """
    assert isinstance(payload, list) and payload, (day, area)
    expected_minutes = 60 if day < date(2025, 10, 1) else 15
    intervals = []
    for item in payload:
        assert {"time_start", "time_end", "SEK_per_kWh"}.issubset(item)
        start = datetime.fromisoformat(item["time_start"])
        end = datetime.fromisoformat(item["time_end"])
        assert start.tzinfo is not None and end.tzinfo is not None
        start_utc, end_utc = start.astimezone(timezone.utc), end.astimezone(timezone.utc)
        expected_end_utc = start_utc + timedelta(minutes=expected_minutes)
        if end_utc != expected_end_utc:
            assert (day == date(2025, 10, 26)
                    and item["time_start"] == "2025-10-26T02:45:00+02:00"
                    and item["time_end"] == "2025-10-26T03:00:00+01:00"
                    and expected_end_utc == datetime(2025, 10, 26, 1, tzinfo=timezone.utc)), (day, area, item)
        intervals.append((start_utc, expected_end_utc))
        assert item["SEK_per_kWh"] is not None
    intervals.sort()
    sweden = ZoneInfo("Europe/Stockholm")
    first = datetime.combine(day, datetime.min.time(), tzinfo=sweden).astimezone(timezone.utc)
    last = datetime.combine(day + timedelta(days=1), datetime.min.time(), tzinfo=sweden).astimezone(timezone.utc)
    assert intervals[0][0] == first and intervals[-1][1] == last, (day, area)
    assert all(left[1] == right[0] for left, right in zip(intervals, intervals[1:])), (day, area)
    assert len(intervals) == (last - first).total_seconds() / (expected_minutes * 60)



In [ ]:
PRICE_START = date(2025, 1, 1)
PRICE_END = datetime.now(ZoneInfo("Europe/Stockholm")).date()
# For a small first run, set PRICE_START = PRICE_END = a past 2026 date.
# Restore these defaults before the full backfill.
# Check existing day/area keys before opening an HTTP session.
price_table = f"{CATALOG}.bronze.electricity_price_raw"
available_tables = {row.tableName for row in spark.sql(f"SHOW TABLES IN {CATALOG}.bronze").collect()}
saved_price_keys = set()
if "electricity_price_raw" in available_tables:
    saved = spark.table(price_table).select(
        "source", "price_date", "price_area", "payload_json"
    ).collect()
    for row in saved:
        key = (row.price_date, row.price_area)
        if (key in saved_price_keys or row.source != "Elpriset just nu" or
                row.price_area not in AREAS or
                not date(2025, 1, 1) <= row.price_date <= datetime.now(ZoneInfo("Europe/Stockholm")).date()):
            raise ValueError(f"{price_table}: invalid/duplicate saved response {key}")
        payload = json.loads(row.payload_json)
        validate_price_payload(payload, *key)
        saved_price_keys.add(key)

expected_keys = {(day, area) for day in days_inclusive(PRICE_START, PRICE_END)
                 for area in AREAS}
missing_price_keys = sorted(expected_keys - saved_price_keys)
if not missing_price_keys:
    print(f"{price_table}: all {len(expected_keys)} target responses valid; 0 API calls, 0 writes")
else:
    pending = []
    with http_session() as session:
        for day, area in missing_price_keys:
            url = f"{PRICE_BASE}/{day:%Y}/{day:%m-%d}_{area}.json"
            context = f"Electricity price; date={day}; area={area}; target={price_table}"
            payload = response_json(session, url, context=context)
            try:
                validate_price_payload(payload, day, area)
            except (AssertionError, KeyError, TypeError, ValueError) as exc:
                raise RuntimeError(
                    f"{context}: price payload validation failed; "
                    f"{type(exc).__name__}: {exc}"
                ) from exc
            pending.append((
                "Elpriset just nu", day, area, url, datetime.now(timezone.utc),
                json.dumps(payload, ensure_ascii=False, separators=(",", ":")),
            ))
            # Commit every 40 responses so interrupted jobs resume near the last key.
            if len(pending) == 40:
                (spark.createDataFrame(pending, PRICE_SCHEMA).write.format("delta")
                      .mode("append").saveAsTable(price_table))
                pending.clear()
            time.sleep(0.2)
    if pending:
        (spark.createDataFrame(pending, PRICE_SCHEMA).write.format("delta")
              .mode("append").saveAsTable(price_table))
    actual_keys = {(r.price_date, r.price_area) for r in
                   spark.table(price_table).select("price_date", "price_area").collect()}
    if not expected_keys <= actual_keys:
        raise ValueError(f"{price_table}: some targeted day/area responses are missing")
    print(f"{price_table}: {len(expected_keys)} responses complete; "
          f"{len(missing_price_keys)} API calls this run")
